# Directory Listing and Template Management

This notebook helps manage the web templates directory by:
1. Creating a directory listing
2. Saving the listing to a Markdown file
3. Updating base.html with the listing
4. Updating server.py as needed

## Import Required Libraries

In [ ]:
# Import necessary libraries
import os
import re
import datetime
import shutil

## List Files in Directory

In [ ]:
# Define the templates directory path
templates_dir = "src/web/templates"

# Get the current directory
current_dir = os.getcwd()
print(f"Current directory: {current_dir}")

# Check if the templates directory exists
templates_path = os.path.join(current_dir, templates_dir)
if os.path.exists(templates_path):
    print(f"Templates directory exists at: {templates_path}")
else:
    print(f"Templates directory not found at: {templates_path}")
    # Create the directory if it doesn't exist
    os.makedirs(templates_path, exist_ok=True)
    print(f"Created templates directory at: {templates_path}")

# List all files in the templates directory
try:
    files = os.listdir(templates_path)
    print(f"Found {len(files)} files in templates directory:")
    
    # Filter HTML files only
    html_files = [f for f in files if f.endswith('.html')]
    print(f"HTML files ({len(html_files)}):")
    for file in html_files:
        print(f"  - {file}")
    
    # Other files
    other_files = [f for f in files if not f.endswith('.html')]
    if other_files:
        print(f"Other files ({len(other_files)}):")
        for file in other_files:
            print(f"  - {file}")
except Exception as e:
    print(f"Error listing files: {e}")

## Write Directory Listing to Markdown File

In [ ]:
# Create a formatted markdown listing of the templates
def create_markdown_listing(files_list, templates_path):
    markdown_content = f"# Templates Directory Listing\n\n"
    markdown_content += f"Generated on: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    
    # Group files by type
    html_files = [f for f in files_list if f.endswith('.html')]
    other_files = [f for f in files_list if not f.endswith('.html')]
    
    # Add HTML files section
    markdown_content += f"## HTML Templates ({len(html_files)})\n\n"
    
    if html_files:
        for file in sorted(html_files):
            file_path = os.path.join(templates_path, file)
            file_size = os.path.getsize(file_path)
            mod_time = os.path.getmtime(file_path)
            mod_time_str = datetime.datetime.fromtimestamp(mod_time).strftime('%Y-%m-%d %H:%M:%S')
            
            # Try to extract title from HTML file
            title = "No title found"
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                    title_match = re.search(r'<title>(.*?)</title>', content)
                    if title_match:
                        title = title_match.group(1)
            except Exception as e:
                title = f"Error reading file: {e}"
                
            markdown_content += f"### {file}\n\n"
            markdown_content += f"- **Title:** {title}\n"
            markdown_content += f"- **Size:** {file_size} bytes\n"
            markdown_content += f"- **Last Modified:** {mod_time_str}\n\n"
    else:
        markdown_content += "No HTML files found.\n\n"
    
    # Add other files section if needed
    if other_files:
        markdown_content += f"## Other Files ({len(other_files)})\n\n"
        for file in sorted(other_files):
            file_path = os.path.join(templates_path, file)
            file_size = os.path.getsize(file_path)
            mod_time = os.path.getmtime(file_path)
            mod_time_str = datetime.datetime.fromtimestamp(mod_time).strftime('%Y-%m-%d %H:%M:%S')
            
            markdown_content += f"- **{file}** ({file_size} bytes, last modified: {mod_time_str})\n"
    
    return markdown_content

# Write the markdown listing to a file
try:
    markdown_content = create_markdown_listing(files, templates_path)
    
    # Define the output markdown file path
    output_file = "templates_listing.md"
    
    # Write to the file
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(markdown_content)
    
    print(f"Successfully wrote templates listing to {output_file}")
    print(f"File size: {os.path.getsize(output_file)} bytes")
except Exception as e:
    print(f"Error writing markdown file: {e}")

## Update base.html

In [ ]:
# Function to update base.html with the templates listing
def update_base_html(base_html_path, templates_listing_path):
    try:
        # Check if base.html exists
        if not os.path.exists(base_html_path):
            print(f"base.html not found at {base_html_path}")
            return False
        
        # Check if templates_listing.md exists
        if not os.path.exists(templates_listing_path):
            print(f"templates_listing.md not found at {templates_listing_path}")
            return False
        
        # Read the templates listing content
        with open(templates_listing_path, 'r', encoding='utf-8') as f:
            templates_content = f.read()
        
        # Read the current base.html content
        with open(base_html_path, 'r', encoding='utf-8') as f:
            base_html_content = f.read()
        
        # Create a backup of the original file
        backup_path = f"{base_html_path}.bak"
        shutil.copy2(base_html_path, backup_path)
        print(f"Created backup of base.html at {backup_path}")
        
        # Look for a specific comment tag where we should insert the template listing
        template_tag_start = "<!-- TEMPLATES_LISTING_START -->"
        template_tag_end = "<!-- TEMPLATES_LISTING_END -->"
        
        if template_tag_start in base_html_content and template_tag_end in base_html_content:
            # Replace the content between the tags
            pattern = f"{template_tag_start}.*?{template_tag_end}"
            replacement = f"{template_tag_start}\n{templates_content}\n{template_tag_end}"
            new_content = re.sub(pattern, replacement, base_html_content, flags=re.DOTALL)
            
            # Write the updated content back to base.html
            with open(base_html_path, 'w', encoding='utf-8') as f:
                f.write(new_content)
            
            print(f"Successfully updated base.html with templates listing")
            return True
        else:
            print(f"Template tags not found in base.html. Please add the following tags where you want to insert the templates listing:")
            print(f"{template_tag_start}")
            print(f"{template_tag_end}")
            return False
            
    except Exception as e:
        print(f"Error updating base.html: {e}")
        return False

# Define paths
base_html_path = os.path.join(templates_path, "base.html")
templates_listing_path = "templates_listing.md"

# Update base.html
update_result = update_base_html(base_html_path, templates_listing_path)

if not update_result:
    print("\nManual instructions for updating base.html:")
    print("1. Open base.html in a text editor")
    print("2. Add the following comment tags where you want to insert the templates listing:")
    print("   <!-- TEMPLATES_LISTING_START -->")
    print("   <!-- TEMPLATES_LISTING_END -->")
    print("3. Run this notebook again to insert the templates listing between these tags")

## Update server.py

In [ ]:
# Function to update server.py to handle the templates
def update_server_py(server_py_path):
    try:
        # Check if server.py exists
        if not os.path.exists(server_py_path):
            print(f"server.py not found at {server_py_path}")
            return False
        
        # Read the current server.py content
        with open(server_py_path, 'r', encoding='utf-8') as f:
            server_content = f.read()
        
        # Create a backup of the original file
        backup_path = f"{server_py_path}.bak"
        shutil.copy2(server_py_path, backup_path)
        print(f"Created backup of server.py at {backup_path}")
        
        # Check if we need to add import statements
        required_imports = [
            "from flask import render_template",
            "import os",
            "import datetime"
        ]
        
        modified = False
        new_content = server_content
        
        # Add any missing imports
        for imp in required_imports:
            if imp not in server_content:
                if "import " in new_content:
                    # Find the last import statement
                    import_lines = re.findall(r'^.*import.*$', new_content, re.MULTILINE)
                    last_import = import_lines[-1] if import_lines else None
                    
                    if last_import:
                        # Insert after the last import
                        new_content = new_content.replace(last_import, f"{last_import}\n{imp}")
                    else:
                        # Add at the beginning of the file
                        new_content = f"{imp}\n{new_content}"
                else:
                    # No imports found, add at the beginning
                    new_content = f"{imp}\n{new_content}"
                modified = True
        
        # Check if we need to add a route for templates listing
        templates_route = """
@app.route('/templates')
def templates_listing():
    """" Display a listing of all templates """
    templates_dir = os.path.join(app.root_path, 'templates')
    
    # Get list of all template files
    templates = []
    if os.path.exists(templates_dir):
        for filename in os.listdir(templates_dir):
            if filename.endswith('.html'):
                file_path = os.path.join(templates_dir, filename)
                mod_time = datetime.datetime.fromtimestamp(os.path.getmtime(file_path))
                templates.append({
                    'name': filename,
                    'path': f'/templates/{filename}',
                    'modified': mod_time.strftime('%Y-%m-%d %H:%M:%S'),
                    'size': os.path.getsize(file_path)
                })
    
    return render_template('templates_listing.html', templates=templates)
"""
        
        if "@app.route('/templates')" not in new_content:
            # Find where to add the route
            if "@app.route" in new_content:
                # Find the last route definition
                route_matches = re.findall(r'@app\.route.*?\ndef .*?:.*?\n', new_content, re.DOTALL)
                if route_matches:
                    last_route = route_matches[-1]
                    # Find where the route function ends
                    route_func_name = re.search(r'def (.*?):', last_route).group(1)
                    route_pattern = f"def {route_func_name}:.*?(?=\n@|\n\n|$)"
                    route_func_match = re.search(route_pattern, new_content, re.DOTALL)
                    
                    if route_func_match:
                        route_end = route_func_match.end()
                        new_content = f"{new_content[:route_end]}\n{templates_route}{new_content[route_end:]}"
                        modified = True
            else:
                # No routes found, add after app initialization
                app_init_pattern = r"app\s*=\s*Flask.*"
                app_init_match = re.search(app_init_pattern, new_content)
                
                if app_init_match:
                    app_init_end = app_init_match.end()
                    new_content = f"{new_content[:app_init_end]}\n\n{templates_route}{new_content[app_init_end:]}"
                    modified = True
                else:
                    # Just append to the end
                    new_content = f"{new_content}\n\n{templates_route}"
                    modified = True
        
        # Write changes if any were made
        if modified:
            with open(server_py_path, 'w', encoding='utf-8') as f:
                f.write(new_content)
            print(f"Updated server.py with necessary changes")
            return True
        else:
            print(f"No changes needed for server.py")
            return True
            
    except Exception as e:
        print(f"Error updating server.py: {e}")
        return False

# Define path to server.py
server_py_path = "src/web/server.py"

# Update server.py
server_update_result = update_server_py(server_py_path)

if not server_update_result:
    print("\nManual instructions for updating server.py:")
    print("1. Open server.py in a text editor")
    print("2. Make sure the following imports are present:")
    print("   from flask import render_template")
    print("   import os")
    print("   import datetime")
    print("3. Add a route for displaying the templates listing")

## Create templates_listing.html Template

If we're going to display the templates listing on the web, we need to create a template for it.

In [ ]:
# Function to create a templates_listing.html file
def create_templates_listing_html(output_path):
    html_content = """
{% extends "base.html" %}

{% block title %}Templates Listing{% endblock %}

{% block content %}
<div class="container mt-4">
    <h1>Templates Listing</h1>
    <p class="text-muted">Generated on: {{ now }}</p>
    
    <div class="card mb-4">
        <div class="card-header">
            <h5 class="mb-0">Available Templates ({{ templates|length }})</h5>
        </div>
        <div class="card-body">
            {% if templates %}
                <div class="table-responsive">
                    <table class="table table-striped table-hover">
                        <thead>
                            <tr>
                                <th>Name</th>
                                <th>Size</th>
                                <th>Last Modified</th>
                                <th>Actions</th>
                            </tr>
                        </thead>
                        <tbody>
                            {% for template in templates %}
                            <tr>
                                <td>{{ template.name }}</td>
                                <td>{{ template.size }} bytes</td>
                                <td>{{ template.modified }}</td>
                                <td>
                                    <a href="{{ template.path }}" class="btn btn-sm btn-outline-primary" target="_blank">
                                        View
                                    </a>
                                </td>
                            </tr>
                            {% endfor %}
                        </tbody>
                    </table>
                </div>
            {% else %}
                <p class="alert alert-info">No templates found.</p>
            {% endif %}
        </div>
    </div>
    
    <div class="card">
        <div class="card-header">
            <h5 class="mb-0">Raw Template Listing</h5>
        </div>
        <div class="card-body">
            <div class="bg-light p-3 rounded">
                <pre><code>{{ templates|pprint }}</code></pre>
            </div>
        </div>
    </div>
</div>
{% endblock %}
"""
    
    try:
        # Ensure the directory exists
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
        # Write the template file
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(html_content)
            
        print(f"Successfully created templates_listing.html at {output_path}")
        return True
    except Exception as e:
        print(f"Error creating templates_listing.html: {e}")
        return False

# Create the templates_listing.html file
templates_listing_html_path = os.path.join(templates_path, "templates_listing.html")
create_templates_listing_html(templates_listing_html_path)

## Summary

This notebook has:

1. Listed all files in the templates directory
2. Created a detailed Markdown listing of the templates
3. Updated (or provided instructions to update) base.html with the listing
4. Updated (or provided instructions to update) server.py to handle the templates listing
5. Created a templates_listing.html file for displaying the templates on the web

To complete the setup, make sure:

1. The Flask server is configured correctly
2. All templates are properly placed in the templates directory
3. The base.html file has the required template tags for the listing